# Exploration, Identification and Evaluation of most relevant features for the detection of malicious Android applications using machine learning models

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing & Selection
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import SelectKBest, chi2, mutual_info_classif
from sklearn.ensemble import ExtraTreesClassifier

# Models
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, f1_score, recall_score

# Configuration
%matplotlib inline
sns.set_theme(style="whitegrid")

## Data import, cleaning, and visualisation

In [ ]:
dataset_path = 'android_malware_dataset.csv'
df = pd.read_csv(dataset_path)

def clean_data(data):
  # Delete features with only one value
  data = data.loc[:, data.nunique() > 1]

  # Fill empty values with 0
  data = data.fillna(0)

  # Encodage target (Malware = 1, Benign = 0)
  le = LabelEncoder()
  data['class'] = le.fit_transform(data['class'])

  return data

df_cleaned = clean_data(df)
print(f"Dimensions after cleaning : {df_cleaned.shape}")

## Features analyze

In [ ]:

# Distribution of classes
plt.figure(figsize=(6, 4))
sns.countplot(x='class', data=df_cleaned)
plt.title('Répartition : Bénin (0) vs Malveillant (1)')
plt.show()

# Heatmap of 10 features on the most correlated to the target
top_corr = df_cleaned.corr()['class'].sort_values(ascending=False).head(11)
sns.heatmap(df_cleaned[top_corr.index].corr(), annot=True, cmap='RdYlGn')
plt.title('Top 10 Features corrélées au Malware')

## Most relevant features identification

In [ ]:
X = df_cleaned.drop('class', axis=1)
y = df_cleaned['class']

# Approach 1 : Get importance via Trees (ExtraTrees)
model_fi = ExtraTreesClassifier(n_estimators=100)
model_fi.fit(X, y)

feat_importances = pd.Series(model_fi.feature_importances_, index=X.columns)
feat_importances.nlargest(15).plot(kind='barh')
plt.title('Top 15 Features discriminantes (ExtraTrees)')
plt.show()

# Approach 2 : Filter statistics (Mutual Information)
# Useful to catch non-linear relations
mi_score = mutual_info_classif(X, y)
mi_series = pd.Series(mi_score, index=X.columns)

## Split dataset for training and test

In [ ]:
# Split : 70% Train, 15% Validation, 15% Test
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# Standardisation (necessary for SVM)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

## Training and evaluations

In [ ]:
models = {
  "Random Forest": RandomForestClassifier(n_estimators=100),
  "SVM": SVC(kernel='linear', probability=True)
}

results = {}

for name, model in models.items():
  # Training
  model.fit(X_train_scaled, y_train)

  # Prediction
  y_pred = model.predict(X_test_scaled)
  
  # Metrics
  results[name] = {
    "F1": f1_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred)
  }
  
  print(f"--- {name} ---")
  print(confusion_matrix(y_test, y_pred))
  print(classification_report(y_test, y_pred))

## Results and discussions

In [ ]:
res_df = pd.DataFrame(results).T
res_df.plot(kind='bar', figsize=(10, 5))
plt.title('F1-Score + Recall')
plt.ylabel('Score')
plt.ylim(0, 1.0)
plt.show()

# Saving features identified as "key features"
important_features = feat_importances.nlargest(20).index.tolist()
print(f"Features recommanded for final model : {important_features}")